# Fox timeline

Regenerates the three data files behind the Fox timeline charts:
`fox_timeline.json` (monthly Fox WNBA article counts, with per-topic and per-player
hit counts and the tooltip sentences), `fox_nba_monthly.json` (the NBA backdrop),
and `trans_articles.csv` (one row per article about trans athletes) in `src/data/`.
Reads the Fox corpora from `data/fox_wnba.csv` and `data/fox_nba.csv`.

In [ ]:
import csv, json, re
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd

DATA = Path("../data")
OUT = Path("../src/data")

csv.field_size_limit(10 ** 9)
fox = list(csv.DictReader((DATA / "fox_wnba.csv").open()))
len(fox)

## Fox WNBA timeline

Articles per month from 2019 on, plus how many each month match each culture-war
topic (and each of three players) in the body or headline. A headline is a poor
witness for why an article counted — the match is usually in the body — so each
topic-month also carries the first readable sentence that actually triggered the
match, for the tooltip.

In [ ]:
# the topic vocabulary lives in data/topic_terms.csv (see notebooks/topics.py)
from topics import topics as load_topics
TOPICS = load_topics()

PLAYER_HL = [
    ("Caitlin Clark", r"caitlin clark|caitlin"),
    ("Sophie Cunningham", r"sophie cunningham|cunningham"),
    ("Brittney Griner", r"brittney griner|griner"),
]

# Fox splices ALL-CAPS links to other stories into the prose
SHOUT_RUN = re.compile(r"\b(?:[A-Z][A-Z'\u2019.&-]{1,}[,:;?!]?\s+){3,}[A-Z][A-Z'\u2019.&-]{1,}[,:;?!]?")


def trim(s, limit=220):
    s = re.sub(r"\s+", " ", s).strip()
    return s if len(s) <= limit else s[:limit].rsplit(" ", 1)[0] + "\u2026"


def proving_sentence(body, title, pattern):
    """The first readable sentence where this topic actually shows up.

    Returns (sentence, proved) — `proved` is False when we fell back to the
    headline, which lets the caller keep looking at other articles that month.
    """
    for candidate in re.split(r"(?<=[.!?])\s+", (body or "").replace("\n", " ")):
        candidate = re.sub(r"\s{2,}", " ", SHOUT_RUN.sub(" ", candidate)).strip(" \u2014-\u2013,;:")
        if not (40 <= len(candidate) <= 400):
            continue
        if re.search(pattern, candidate, re.I):
            return trim(candidate, 230), True
    return trim(title or "", 230), bool(re.search(pattern, title or "", re.I))

In [ ]:
months = defaultdict(int)
example = {}
mtopics = {t: defaultdict(int) for t, _ in TOPICS + PLAYER_HL}
mtopic_ex = {t: {} for t, _ in TOPICS + PLAYER_HL}   # topic -> month -> proving sentence
mtopic_hed = {t: {} for t, _ in TOPICS + PLAYER_HL}  # topic -> month -> its headline
mtopic_ok = {t: set() for t, _ in TOPICS + PLAYER_HL}  # where the sentence really proves it

for r in fox:
    d = (r.get("date") or "")[:7]
    # the study window starts in 2019; tag-feed stragglers from earlier
    # would stretch the timeline axis into near-empty years
    if not d or len(d) < 7 or d < "2019":
        continue
    months[d] += 1
    if d not in example:
        example[d] = r.get("headline", "")
    text = ((r.get("body") or "") + " " + (r.get("headline") or "")).lower()
    for t, pat in TOPICS + PLAYER_HL:
        if re.search(pat, text):
            mtopics[t][d] += 1
            # keep looking through the month until one article yields a
            # sentence that actually demonstrates the topic
            if d not in mtopic_ok[t]:
                sentence, proved = proving_sentence(r.get("body"), r.get("headline"), pat)
                if proved or d not in mtopic_ex[t]:
                    mtopic_ex[t][d] = sentence
                    mtopic_hed[t][d] = trim(r.get("headline") or "", 120)
                if proved:
                    mtopic_ok[t].add(d)

# dense month axis from first to last
def next_month(m):
    y, mo = int(m[:4]), int(m[5:7])
    return f"{y + (mo == 12)}-{(mo % 12) + 1:02d}"

ms = sorted(months)
axis, cur = [], ms[0]
while cur <= ms[-1]:
    axis.append({"month": cur, "n": months.get(cur, 0), "ex": example.get(cur, ""),
                 "t": [mtopics[t].get(cur, 0) for t, _ in TOPICS + PLAYER_HL],
                 "tex": [mtopic_ex[t].get(cur, "") for t, _ in TOPICS + PLAYER_HL],
                 "thed": [mtopic_hed[t].get(cur, "") for t, _ in TOPICS + PLAYER_HL]})
    cur = next_month(cur)

with (OUT / "fox_timeline.json").open("w") as f:
    json.dump({"months": axis, "total": sum(months.values()),
               "topics": [t for t, _ in TOPICS],
               "players": [t for t, _ in PLAYER_HL]}, f)
print(f"fox_timeline: {len(axis)} months, {sum(months.values())} articles, {ms[0]}..{ms[-1]}")

pd.DataFrame([{"month": m["month"], "articles": m["n"],
               **{t: m["t"][i] for i, (t, _) in enumerate(TOPICS[:3] + PLAYER_HL)}}
              for m in axis]).tail(8)

## Fox NBA backdrop

Fox NBA articles per month from `data/fox_nba.csv`, aligned to the WNBA timeline's
month axis so the chart can join on index. `from` records the first month with any
NBA data (the corpus reaches back to 2016, before the chart's axis starts).

In [ ]:
nba = list(csv.DictReader((DATA / "fox_nba.csv").open()))
by_month = Counter(r["date"][:7] for r in nba if len(r.get("date") or "") >= 7)
month_axis = [m["month"] for m in axis]

with (OUT / "fox_nba_monthly.json").open("w") as f:
    json.dump({"from": min(by_month),
               "months": month_axis,
               "counts": [by_month.get(m, 0) for m in month_axis]}, f)

on_axis = sum(by_month.get(m, 0) for m in month_axis)
print(f"{len(nba):,} NBA articles; earliest month {min(by_month)}; {on_axis:,} fall on the chart's axis")
pd.DataFrame({"month": month_axis, "nba": [by_month.get(m, 0) for m in month_axis]}).tail(8)

## Trans-coverage unit chart

One row per Fox WNBA article matching the timeline's "transgender / trans" pattern
in the headline or body, sorted by date.

In [ ]:
TRANS = re.compile(TOPICS[0][1], re.I)

rows = []
for r in fox:
    date = (r.get("date") or "")[:10]
    if len(date) < 10:
        continue
    text = (r.get("headline") or "") + " " + (r.get("body") or "")
    if TRANS.search(text):
        rows.append({"date": date, "month": date[:7],
                     "title": (r.get("headline") or "")[:160],
                     "url": r.get("url", "")})
rows.sort(key=lambda a: a["date"])

with (OUT / "trans_articles.csv").open("w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["date", "month", "title", "url"])
    w.writeheader()
    w.writerows(rows)

print(f"trans_articles: {len(rows)} articles, {rows[0]['date']}..{rows[-1]['date']}")
pd.DataFrame(rows)[["date", "title"]].tail(8)